# Stage 1 — Warmup Training (Kaggle)

Notebook chạy Stage 1 training trên Kaggle.

**Setup yêu cầu:**
1. Enable GPU (Settings → Accelerator → GPU P100 hoặc T4 x2).
2. Enable Internet.
3. **Add Data** (sidebar bên phải):
   - `arnaud58/flickrfaceshq-dataset-ffhq` (FFHQ — Stage 1 training)
   - hoặc Asian face dataset khác bạn chọn
4. **Upload code repo** (Add Data → Upload → upload zip vn-old-portrait-restoration.zip,
   hoặc clone từ GitHub nếu đã push).

In [ ]:
# 1. Setup code
import os, sys

# Option A: nếu upload zip vào Kaggle dataset
# !cp /kaggle/input/vn-old-portrait/vn-old-portrait-restoration.zip /kaggle/working/
# !cd /kaggle/working && unzip -q vn-old-portrait-restoration.zip
# %cd /kaggle/working/vn-old-portrait-restoration

# Option B: clone từ GitHub (nếu đã push)
!git clone https://github.com/<your-username>/vn-old-portrait-restoration.git /kaggle/working/repo
%cd /kaggle/working/repo

# Add to Python path
sys.path.insert(0, '/kaggle/working/repo')
print('CWD:', os.getcwd())

In [ ]:
# 2. Install dependencies
!pip install -q einops pyyaml tensorboard wandb facexlib insightface mediapipe lpips pyiqa

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 3. Setup paths cho Kaggle
# FFHQ dataset trên Kaggle thường ở /kaggle/input/<slug>/
# Kiểm tra path thực tế:
!ls /kaggle/input/

In [ ]:
# 4. Modify config — đổi path cho phù hợp với Kaggle
import yaml

with open('configs/stage1_warmup.yaml') as f:
    config = yaml.safe_load(f)

# Đổi path data
# Ví dụ nếu FFHQ ở /kaggle/input/flickrfaceshq-dataset-ffhq/thumbnails128x128/
config['data']['hq_dirs'] = [
    '/kaggle/input/flickrfaceshq-dataset-ffhq/thumbnails128x128',
]
# Val tạm dùng subset của FFHQ — sau này tải CelebA-HQ test thì đổi
config['data']['val_dir'] = '/kaggle/input/flickrfaceshq-dataset-ffhq/thumbnails128x128'

# Kaggle session 9h → giảm total_iters nếu test
# Full run: 30k iters
config['train']['total_iters'] = 30000
config['data']['image_size'] = 256        # giảm từ 512 → 256 để fit T4 15GB và thumbnails 128 không upscale nhiều
config['data']['batch_size'] = 16          # tăng batch khi image size nhỏ
config['data']['num_workers'] = 2

# Output dir trên Kaggle /kaggle/working (persistent)
config['log']['output_dir'] = '/kaggle/working/checkpoints/stage1'
config['log']['tensorboard_dir'] = '/kaggle/working/tensorboard/stage1'

# W&B (optional)
config['log']['wandb']['enable'] = False   # set True nếu đã setup wandb login

print(yaml.dump(config, default_flow_style=False))

In [ ]:
# 5. (Optional) W&B login
# import wandb; wandb.login(key='your_wandb_api_key')

In [ ]:
# 6. Test data loader nhanh (verify path đúng + ảnh load được)
from src.utils.data_loader import build_train_dataset

ds = build_train_dataset(config)
print(f'Dataset size: {len(ds)}')
sample = ds[0]
print(f'LQ: {sample["lq"].shape}')
print(f'HQ: {sample["hq"].shape}')
print(f'deg_vec: {sample["deg_vec"].numpy().round(2).tolist()}')

# Visualize
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    s = ds[i]
    axes[0, i].imshow(s['hq'].permute(1, 2, 0).numpy())
    axes[0, i].set_title('HQ'); axes[0, i].axis('off')
    axes[1, i].imshow(s['lq'].permute(1, 2, 0).numpy())
    axes[1, i].set_title(f'LQ vec={s["deg_vec"].numpy().round(1).tolist()}', fontsize=8)
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Train Stage 1
from src.training.trainer_stage1 import Stage1Trainer

trainer = Stage1Trainer(config)
trainer.train()

In [ ]:
# 8. Xem TensorBoard logs
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/tensorboard

In [ ]:
# 9. Verify checkpoint saved
!ls -lh /kaggle/working/checkpoints/stage1/